# 06: Evaluation theory and experiment design

[Course index](../README_en.md) | [中文版本](../notebooks/06_evaluation.ipynb)

**Goal:** choose metrics that answer a research question instead of searching for one universal LLM score.


## 1. Three levels of evidence

Ask separately whether language modeling is retained, auditable instructions are completed, and preference optimization improves external behavior rather than only its proxy.


### Hands-on check

Import the production metric functions used below.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'model').exists(): ROOT = ROOT.parents[2]
sys.path.insert(0, str(ROOT))
from eval.metrics import compute_all_metrics, compute_keyword_coverage


## 2. Likelihood metrics

$$NLL=-rac1N\sum_t\log p(x_t|x_{<t}),\qquad PPL=e^{NLL}.$$

LM PPL detects forgetting; response NLL measures reference-answer likelihood. Neither directly measures generated task success.


### Hands-on check

Convert an example NLL to perplexity.


In [ ]:
from eval.metrics import compute_perplexity

nll = 1.7
ppl = compute_perplexity(nll)
print(f"NLL={nll:.2f}, PPL={ppl:.3f}")
assert abs(ppl - 5.473947) < 1e-5


## 3. Auditable task metrics

QA exact match normalizes case/punctuation. Keyword coverage is $|K\cap tokens(\hat y)|/|K|$ and all-pass requires coverage 1. Sentence exact checks whether generated sentence count equals the request.


### Hands-on check

Normalize two equivalent short answers and compute exact match.


In [ ]:
import re

def normalize_answer(text):
    return " ".join(re.findall(r"[a-z0-9]+", text.lower()))

prediction, reference = "Lily.", "lily"
exact_match = float(normalize_answer(prediction) == normalize_answer(reference))
print("normalized prediction:", normalize_answer(prediction))
print("exact match:", exact_match)
assert exact_match == 1.0


### Hands-on check

Compute keyword coverage on controlled examples.


In [ ]:
texts = ['A bird sat near the blue river.', 'Only a bird appears.']
keywords = [['bird', 'blue', 'river'], ['bird', 'blue', 'river']]
print('mean keyword coverage:', compute_keyword_coverage(texts, keywords))


## 4. Preference metrics

The reported policy margin is

$$m_	heta=\log\pi_	heta(y_w|x)-\log\pi_	heta(y_l|x).$$

DPO loss additionally uses $\Delta m=m_	heta-m_{ref}$. Do not mix raw ranking accuracy with reference-adjusted improvement.


### Hands-on check

Compute both raw and reference-adjusted margins.


In [ ]:
policy_chosen, policy_rejected = -1.4, -2.0
ref_chosen, ref_rejected = -1.5, -1.8
policy_margin = policy_chosen - policy_rejected
relative_margin = policy_margin - (ref_chosen - ref_rejected)
print("policy margin:", policy_margin)
print("relative-to-reference margin:", relative_margin)
assert policy_margin > 0 and relative_margin > 0


## 5. Generation proxies

Distinct-$n$, repetition, ending rate, and length are useful diagnostics but not semantic judges. High diversity can be random; ending punctuation can be confidently wrong.


### Hands-on check

Compare normal text and repetition using production metrics.


In [ ]:
good = ['The red bird flew home. The child smiled.']
repeat = ['bird bird bird bird bird bird bird']
print('normal:', compute_all_metrics(good, loss=1.7))
print('repeat:', compute_all_metrics(repeat, loss=1.7))


## 6. Experimental discipline

Select checkpoints on validation data, reserve test data for the final report, reuse prompts/seeds/decoding across models, and include Base, ContinuationSFT, and InstructionSFT controls. Report failures and sample size.

- [Previous](05_alignment.ipynb) · [Index](../README_en.md) · [Next](07_reproduce.ipynb)
- [`eval/comprehensive_eval.py`](../../../eval/comprehensive_eval.py)
- [`eval/metrics.py`](../../../eval/metrics.py)
